In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🛡️ GraphQL Security & Query Optimization Guide

**Securing GraphQL APIs and optimizing query performance**

This guide covers:
- GraphQL attack vectors
- Query complexity analysis
- Rate limiting and throttling
- Introspection controls
- Input validation
- Query depth and breadth limits
- Caching strategies
- Cost-based query analysis
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. GraphQL Security Fundamentals

### Query Complexity & Depth Limiting
```python
from dataclasses import dataclass
from typing import Dict, Optional, Any
from enum import Enum

@dataclass
class QueryComplexity:
    """Query complexity metrics"""
    total_complexity: float
    depth: int
    breadth: int
    field_count: int
    is_valid: bool
    violation_reason: Optional[str] = None

class ComplexityAnalyzer:
    """Analyze GraphQL query complexity"""
    
    def __init__(self, max_complexity: float = 1000, max_depth: int = 10, max_fields: int = 100):
        self.max_complexity = max_complexity
        self.max_depth = max_depth
        self.max_fields = max_fields
        self.field_costs: Dict[str, float] = {
            'simple_field': 1.0,
            'list_field': 2.0,
            'connection_field': 5.0,
            'mutation': 10.0
        }
    
    def analyze_query(self, query: str, schema: Dict) -> QueryComplexity:
        """Analyze query complexity"""
        
        # Parse query
        parsed = self._parse_query(query)
        
        # Calculate metrics
        complexity = self._calculate_complexity(parsed, schema)
        depth = self._calculate_depth(parsed)
        breadth = self._calculate_breadth(parsed)
        field_count = self._count_fields(parsed)
        
        # Validate
        is_valid = True
        violation_reason = None
        
        if complexity > self.max_complexity:
            is_valid = False
            violation_reason = f"Complexity {complexity} exceeds max {self.max_complexity}"
        
        if depth > self.max_depth:
            is_valid = False
            violation_reason = f"Depth {depth} exceeds max {self.max_depth}"
        
        if field_count > self.max_fields:
            is_valid = False
            violation_reason = f"Field count {field_count} exceeds max {self.max_fields}"
        
        return QueryComplexity(
            total_complexity=complexity,
            depth=depth,
            breadth=breadth,
            field_count=field_count,
            is_valid=is_valid,
            violation_reason=violation_reason
        )
    
    def _parse_query(self, query: str) -> Dict:
        """Parse GraphQL query"""
        # Simplified parsing
        return {'query': query}
    
    def _calculate_complexity(self, parsed: Dict, schema: Dict) -> float:
        """Calculate query complexity"""
        # Sum costs of all fields
        complexity = 0.0
        for field_type in ['simple_field', 'list_field']:
            if field_type in parsed.get('query', ''):
                complexity += self.field_costs.get(field_type, 1.0)
        return complexity
    
    def _calculate_depth(self, parsed: Dict) -> int:
        """Calculate query depth"""
        # Simplified - count nesting levels
        return len(parsed.get('query', '').split('{'))
    
    def _calculate_breadth(self, parsed: Dict) -> int:
        """Calculate query breadth"""
        # Simplified - count fields at each level
        return len([x for x in parsed.get('query', '').split() if x == ','])
    
    def _count_fields(self, parsed: Dict) -> int:
        """Count total fields"""
        return len([x for x in parsed.get('query', '').split() if x not in ['{', '}', ',']])

class RateLimiter:
    """Rate limit GraphQL queries"""
    
    def __init__(self, max_queries_per_minute: int = 60):
        self.max_queries_per_minute = max_queries_per_minute
        self.client_queries: Dict[str, list] = {}
    
    def is_allowed(self, client_id: str) -> bool:
        """Check if client can execute query"""
        
        from datetime import datetime, timedelta
        
        now = datetime.utcnow()
        minute_ago = now - timedelta(minutes=1)
        
        if client_id not in self.client_queries:
            self.client_queries[client_id] = []
        
        # Clean old queries
        self.client_queries[client_id] = [
            ts for ts in self.client_queries[client_id] if ts > minute_ago
        ]
        
        # Check limit
        if len(self.client_queries[client_id]) < self.max_queries_per_minute:
            self.client_queries[client_id].append(now)
            return True
        
        return False
```

### Input Validation
```python
class InputValidator:
    """Validate GraphQL input"""
    
    def __init__(self):
        self.string_max_length = 1000
        self.list_max_size = 100
    
    def validate_string(self, value: str) -> bool:
        """Validate string input"""
        if not isinstance(value, str):
            return False
        if len(value) > self.string_max_length:
            return False
        return True
    
    def validate_list(self, value: list) -> bool:
        """Validate list input"""
        if not isinstance(value, list):
            return False
        if len(value) > self.list_max_size:
            return False
        return True
    
    def validate_email(self, email: str) -> bool:
        """Validate email input"""
        if '@' not in email or '.' not in email:
            return False
        return True
    
    def validate_id(self, id_value: str) -> bool:
        """Validate ID input"""
        # Alphanumeric and underscore only
        return id_value.replace('_', '').isalnum()
    
    def sanitize_input(self, value: str) -> str:
        """Sanitize input"""
        # Remove dangerous characters
        dangerous_chars = ['<', '>', '"', "'", '&']
        for char in dangerous_chars:
            value = value.replace(char, '')
        return value
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Introspection & Caching

### Introspection Controls
```python
class IntrospectionController:
    """Control schema introspection"""
    
    def __init__(self):
        self.introspection_enabled = True
        self.public_schema_only = False
    
    def handle_introspection_query(self, user_authenticated: bool, user_role: str) -> bool:
        """Allow/deny introspection"""
        
        if not self.introspection_enabled:
            return False
        
        # Public schema only for unauthenticated users
        if not user_authenticated and self.public_schema_only:
            return False
        
        # Admin can always introspect
        if user_role == 'admin':
            return True
        
        # Authenticated users can introspect
        if user_authenticated:
            return True
        
        return False

class QueryCaching:
    """Cache GraphQL query results"""
    
    def __init__(self):
        self.cache: Dict[str, tuple] = {}
        self.cache_ttl: Dict[str, int] = {}
    
    def cache_query(self, query_hash: str, result: Any, ttl_seconds: int = 60):
        """Cache query result"""
        
        from datetime import datetime, timedelta
        
        expires_at = datetime.utcnow() + timedelta(seconds=ttl_seconds)
        self.cache[query_hash] = (result, expires_at)
        self.cache_ttl[query_hash] = ttl_seconds
    
    def get_cached_query(self, query_hash: str) -> Optional[Any]:
        """Get cached query result"""
        
        from datetime import datetime
        
        if query_hash not in self.cache:
            return None
        
        result, expires_at = self.cache[query_hash]
        
        if datetime.utcnow() > expires_at:
            del self.cache[query_hash]
            return None
        
        return result
    
    def invalidate_cache(self, pattern: str = None):
        """Invalidate cache"""
        
        if not pattern:
            self.cache.clear()
            return
        
        # Invalidate matching patterns
        keys_to_delete = [k for k in self.cache.keys() if pattern in k]
        for key in keys_to_delete:
            del self.cache[key]
```

### Batch Query Protection
```python
class BatchQueryProtection:
    """Prevent batch query attacks"""
    
    def __init__(self, max_batch_size: int = 10):
        self.max_batch_size = max_batch_size
    
    def validate_batch(self, queries: list) -> bool:
        """Validate batch queries"""
        
        if not isinstance(queries, list):
            return False
        
        if len(queries) > self.max_batch_size:
            return False
        
        # Each query should be valid
        for query in queries:
            if not isinstance(query, str):
                return False
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Performance Optimization

### Query Performance
```python
@dataclass
class QueryMetrics:
    """Query performance metrics"""
    query_hash: str
    execution_time_ms: float
    resolver_calls: int
    database_queries: int
    cache_hits: int

class QueryOptimizer:
    """Optimize GraphQL query performance"""
    
    def __init__(self):
        self.query_metrics: Dict[str, QueryMetrics] = {}
        self.slow_query_threshold_ms = 100
    
    def record_query_execution(self, query_hash: str, execution_time_ms: float, 
                              resolver_calls: int, db_queries: int, cache_hits: int):
        """Record query execution"""
        
        metrics = QueryMetrics(
            query_hash=query_hash,
            execution_time_ms=execution_time_ms,
            resolver_calls=resolver_calls,
            database_queries=db_queries,
            cache_hits=cache_hits
        )
        
        self.query_metrics[query_hash] = metrics
    
    def get_slow_queries(self) -> list:
        """Get slow queries"""
        
        return [
            m for m in self.query_metrics.values()
            if m.execution_time_ms > self.slow_query_threshold_ms
        ]
    
    def suggest_optimization(self, query_metrics: QueryMetrics) -> str:
        """Suggest optimization"""
        
        if query_metrics.database_queries > query_metrics.resolver_calls:
            return "Too many N+1 queries - use batch loading"
        
        if query_metrics.cache_hits == 0 and query_metrics.execution_time_ms > 50:
            return "Consider caching this query"
        
        return "Query performance acceptable"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. GraphQL Security Checklist

✅ **Query Protection**
- [ ] Complexity analysis implemented
- [ ] Depth limits enforced
- [ ] Breadth limits set
- [ ] Query validation working
- [ ] Rate limiting active

✅ **Input Validation**
- [ ] String length limits
- [ ] List size limits
- [ ] Email validation
- [ ] ID validation
- [ ] Input sanitization

✅ **Caching Strategy**
- [ ] Query caching enabled
- [ ] TTL configured
- [ ] Cache invalidation
- [ ] Batch protection
- [ ] Performance monitored

✅ **Security Controls**
- [ ] Introspection controlled
- [ ] Schema exposure managed
- [ ] Authentication required
- [ ] Authorization enforced
- [ ] Audit logging

✅ **Operational**
- [ ] Slow queries identified
- [ ] Optimization suggestions
- [ ] Documentation
- [ ] Team training
- [ ] Monitoring alerts
</VSCode.Cell>
```